# EXP-0001-CAL-001: Probability Calibration & Reliability Analysis

**Experiment ID:** `EXP-0001-CAL-001`  
**Model Family:** Driver-Only Pairwise Elo Baseline (`EXP-0001-R3` / `EXP-0001-R2`)  
**Hypothesis Reference:** [HYP-0004: Checkpoint Calibration Drift](../docs/research/HYPOTHESES.md#hyp-0004-checkpoint-calibration-drift)  
**Scientific Question (RQ3):** Are candidate-level race-winner probabilities statistically well-calibrated, and does calibration degrade under technical regulation transitions or across seasons?  
**Evaluation Protocol:** Out-of-sample chronological walk-forward predictions (2010–2026, 346 races, 7,285 candidate records)  
**Primary Source Input:** `artifacts/reports/EXP-0001-R3/candidate_predictions.csv`  
**Target Output Directory:** `artifacts/reports/EXP-0001-CAL-001/`  

---

## Research Objectives & Context

Probabilistic forecasting requires not merely ranking competitors accurately, but outputting probability estimates that reflect true empirical frequencies. In Formula 1 forecasting:
1. When a model predicts a driver has an $x\%$ probability of winning, does the driver win in approximately $x\%$ of such instances?
2. Latent rating models (e.g. Bradley-Terry / Luce softmax over Elo ratings) often suffer from structural miscalibration in multi-candidate settings.
3. This notebook establishes the verified baseline calibration metrics (Expected Calibration Error, Maximum Calibration Error, Calibration-in-the-Large, reliability diagrams, and race-clustered bootstrap uncertainty intervals) before evaluating post-hoc recalibration models in future milestones.

## 1. Environment Setup & Robust Path Resolution

We initialize all required dependencies, robustly locate the project root by traversing parent directories for `AGENTS.md` and `README.md`, set up the target output directory, and pre-specify fixed bin edges.

In [1]:
# 1. Environment Setup & Robust Project-Root Discovery
import os
import sys
import json
import math
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Robust project-root discovery by searching parent directories for markers
cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / "AGENTS.md").is_file() and (p / "README.md").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate APEX project root containing AGENTS.md and README.md.")

# Define input source paths (prefer outcome-independent R3, fallback to R2)
R3_CANDIDATES_PATH = PROJECT_ROOT / "artifacts/reports/EXP-0001-R3/candidate_predictions.csv"
R2_CANDIDATES_PATH = PROJECT_ROOT / "artifacts/reports/EXP-0001-R2/candidate_predictions.csv"

if R3_CANDIDATES_PATH.is_file():
    SOURCE_PATH = R3_CANDIDATES_PATH
    SOURCE_VERSION = "EXP-0001-R3"
elif R2_CANDIDATES_PATH.is_file():
    SOURCE_PATH = R2_CANDIDATES_PATH
    SOURCE_VERSION = "EXP-0001-R2"
    print("WARNING: R3 candidate predictions not found; falling back to R2.")
else:
    raise FileNotFoundError("No candidate_predictions.csv found in R3 or R2 reports directories.")

# Output artifact directory
CAL_DIR = PROJECT_ROOT / "artifacts/reports/EXP-0001-CAL-001"
CAL_DIR.mkdir(parents=True, exist_ok=True)

# Pre-specified, fixed probability bin edges
# Half-open intervals: [b_i, b_{i+1}) with the final bin closing at 1.0
BIN_EDGES = np.array([
    0.00, 0.02, 0.04, 0.06, 0.08, 0.10,
    0.15, 0.20, 0.30, 0.50, 1.000000001
])

print(f"Project root discovered: {PROJECT_ROOT}")
print(f"Source candidate dataset: {SOURCE_PATH.relative_to(PROJECT_ROOT)} ({SOURCE_VERSION})")
print(f"Calibration output directory: {CAL_DIR.relative_to(PROJECT_ROOT)}")
print(f"Pre-specified bin edges ({len(BIN_EDGES) - 1} bins): {BIN_EDGES.tolist()}")

Project root discovered: /Users/dhruv/dhruv files/apex-f1
Source candidate dataset: artifacts/reports/EXP-0001-R3/candidate_predictions.csv (EXP-0001-R3)
Calibration output directory: artifacts/reports/EXP-0001-CAL-001
Pre-specified bin edges (10 bins): [0.0, 0.02, 0.04, 0.06, 0.08, 0.1, 0.15, 0.2, 0.3, 0.5, 1.000000001]


## 2. Load Candidate Predictions & Schema Normalization

We load the verified candidate predictions, dynamically identify the actual winner label column across candidate schema variants (`actual_winner`, `is_actual_winner`, `is_winner`), coerce `cal["y"]` into a clean binary integer $\{0, 1\}$, and convert `cal["win_probability"]` to float numeric.

In [2]:
# 2. Load Data & Normalize Label Column
cal = pd.read_csv(SOURCE_PATH)

# Dynamically detect winner-label column
label_col = next(
    (col for col in ["actual_winner", "is_actual_winner", "is_winner", "winner"] if col in cal.columns),
    None,
)
if label_col is None:
    raise ValueError(f"Could not identify winner-label column. Available columns: {cal.columns.tolist()}")

# Normalize binary target y
cal["y"] = cal[label_col]
if cal["y".strip()].dtype == object:
    normalized = cal["y"].astype(str).str.strip().str.lower()
    boolean_map = {"true": 1, "false": 0, "1": 1, "0": 0}
    if set(normalized.dropna().unique()).issubset(boolean_map):
        cal["y"] = normalized.map(boolean_map)
    else:
        cal["y"] = pd.to_numeric(cal["y"], errors="coerce")
else:
    cal["y"] = pd.to_numeric(cal["y"], errors="coerce")

cal["y"] = cal["y"].astype(int)
cal["win_probability"] = pd.to_numeric(cal["win_probability"], errors="coerce")

required_fields = ["season", "round", "driver_id", "y", "win_probability", "regulation_era"]
missing = [c for c in required_fields if c not in cal.columns]
if missing:
    raise ValueError(f"Missing required columns in candidate predictions: {missing}")

print(f"Successfully loaded {len(cal)} candidate records across {cal['season'].nunique()} seasons.")
print(f"Winner label column identified: '{label_col}' -> mapped to cal['y']")
print(f"Total race winners observed in dataset: {cal['y'].sum()}")

Successfully loaded 7285 candidate records across 17 seasons.
Winner label column identified: 'actual_winner' -> mapped to cal['y']
Total race winners observed in dataset: 346


## 3. Probability Integrity Checks & Axiomatic Verification

Before evaluating calibration, we assert fundamental probabilistic axioms:
1. **Boundedness:** Every candidate probability $P_i \in [0, 1]$.
2. **Simplex Normalization:** For every Grand Prix $k$, candidate probabilities sum to one within numerical tolerance: $\left| \sum_{i=1}^{M_k} P_{ik} - 1.0 \right| < 10^{-5}$.
3. **Exact Single Winner:** Exactly one candidate per race has the positive winner label ($y = 1$).
4. **Uniqueness:** Entrant keys `(season, round, driver_id)` are strictly unique (no duplicate entrant records per race).
5. **Completeness:** No nulls or NaNs in required prediction fields.

> **Conceptual Distinction:** Candidate-level one-vs-rest metrics evaluate individual driver win predictions against binary outcomes ($y_i \in \{0, 1\}$), which is mathematically distinct from race-level multiclass scoring rules reported in EXP-0001-R3.

In [3]:
# 3. Run Probability & Data Integrity Assertions
assert cal[required_fields].notna().all().all(), "Integrity Error: Found NaN/null values in required fields!"
assert cal["y"].isin([0, 1]).all(), "Integrity Error: Outcome labels must be strictly binary {0, 1}!"
assert cal["win_probability"].between(0.0, 1.0).all(), "Integrity Error: Probabilities must lie in [0, 1]!"

# Verify uniqueness of candidate entrant keys
assert not cal.duplicated(["season", "round", "driver_id"]).any(), "Integrity Error: Found duplicate driver records in a race!"

# Race-level aggregation checks
race_key = ["season", "round"]
per_race = cal.groupby(race_key).agg(
    candidates=("driver_id", "size"),
    winner_count=("y", "sum"),
    probability_sum=("win_probability", "sum"),
)

assert per_race["winner_count"].eq(1).all(), "Integrity Error: Every race must have exactly one winner!"

max_sum_error = float((per_race["probability_sum"] - 1.0).abs().max())
assert max_sum_error < 1e-5, f"Integrity Error: Race probability sum error ({max_sum_error}) exceeds tolerance!"

print("All probability axioms and integrity assertions verified successfully:")
print(f"  - Total Races Analyzed    : {len(per_race)}")
print(f"  - Total Candidate Records : {len(cal)}")
print(f"  - Max Probability-Sum Dev : {max_sum_error:.2e} (< 1e-5)")
print(f"  - Candidate Field Size    : min={per_race['candidates'].min()}, mean={per_race['candidates'].mean():.1f}, max={per_race['candidates'].max()}")

All probability axioms and integrity assertions verified successfully:
  - Total Races Analyzed    : 346
  - Total Candidate Records : 7285
  - Max Probability-Sum Dev : 1.55e-15 (< 1e-5)
  - Candidate Field Size    : min=18, mean=21.1, max=24


## 4. Binned Reliability & Proper Calibration Metrics

### Mathematical Definitions & Empty-Bin Treatment
Let $N$ be the total candidate records ($N = 7,285$). Candidate predictions $\hat{p}_i$ are partitioned into $B = 10$ half-open bins $[b_j, b_{j+1})$ using `np.digitize`.

**Critical Methodological Requirement:**
- In Formula 1 race-winner modeling, probabilities rarely exceed $0.30$. Bins such as $[0.30, 0.50)$ and $[0.50, 1.00]$ may contain **zero** candidates.
- **Empty bins must NOT contribute to ECE or MCE.** Assigning artificial midpoints or zero frequencies to empty bins produces synthetic gaps that distort calibration error metrics.
- ECE is computed as the candidate-count-weighted absolute calibration gap strictly over non-empty bins:
  $$\text{ECE} = \sum_{b \in B_{\text{non-empty}}} \frac{|B_b|}{N} |\bar{y}_b - \bar{p}_b|$$
- MCE is computed as the maximum absolute calibration gap across non-empty bins:
  $$\text{MCE} = \max_{b \in B_{\text{non-empty}}} |\bar{y}_b - \bar{p}_b|$$
- **Calibration-in-the-Large (CITL):** Global mean observed frequency minus global mean predicted probability:
  $$\text{CITL} = \bar{y} - \bar{p} = \frac{1}{N} \sum_{i=1}^{N} y_i - \frac{1}{N} \sum_{i=1}^{N} \hat{p}_i$$
- **Candidate One-vs-Rest Brier Score:** $\text{BS}_{\text{cand}} = \frac{1}{N} \sum_{i=1}^N (\hat{p}_i - y_i)^2$
- **Candidate One-vs-Rest Log Loss:** $\text{LL}_{\text{cand}} = -\frac{1}{N} \sum_{i=1}^N [y_i \ln(\hat{p}_i) + (1 - y_i) \ln(1 - \hat{p}_i)]$

In [4]:
# 4. Compute Calibration Table & Proper Scoring Metrics
p = cal["win_probability"].to_numpy(dtype=float)
y = cal["y"].to_numpy(dtype=int)

def compute_calibration_table(p_arr, y_arr, edges=BIN_EDGES):
    """
    Compute binned calibration metrics, properly segregating non-empty bins from empty bins.
    """
    n_total = len(p_arr)
    bin_id = np.digitize(p_arr, edges[1:-1], right=False)
    
    rows = []
    for i in range(len(edges) - 1):
        mask = (bin_id == i)
        count = int(mask.sum())
        b_lower = float(edges[i])
        b_upper = float(min(edges[i + 1], 1.0))
        label = f"[{b_lower:.2f}, {b_upper:.2f})"
        
        if count > 0:
            mean_pred = float(p_arr[mask].mean())
            obs_freq = float(y_arr[mask].mean())
            abs_gap = float(abs(obs_freq - mean_pred))
            weight = float(count / n_total)
            rows.append({
                "bin_index": i,
                "bin_lower": b_lower,
                "bin_upper": b_upper,
                "bin_label": label,
                "n_candidates": count,
                "weight": weight,
                "mean_predicted_probability": mean_pred,
                "observed_win_frequency": obs_freq,
                "calibration_gap": float(obs_freq - mean_pred),
                "absolute_gap": abs_gap,
                "is_empty": False,
            })
        else:
            rows.append({
                "bin_index": i,
                "bin_lower": b_lower,
                "bin_upper": b_upper,
                "bin_label": label,
                "n_candidates": 0,
                "weight": 0.0,
                "mean_predicted_probability": np.nan,
                "observed_win_frequency": np.nan,
                "calibration_gap": np.nan,
                "absolute_gap": np.nan,
                "is_empty": True,
            })
            
    df_all = pd.DataFrame(rows)
    assert df_all["n_candidates"].sum() == n_total, "Integrity Violation: Bin counts do not sum to total candidates!"
    
    # Non-empty bins for ECE and MCE
    df_non_empty = df_all[~df_all["is_empty"]].copy()
    ece = float((df_non_empty["weight"] * df_non_empty["absolute_gap"]).sum())
    mce = float(df_non_empty["absolute_gap"].max())
    citl = float(y_arr.mean() - p_arr.mean())
    
    # Candidate one-vs-rest binary proper scoring rules
    eps = 1e-15
    p_safe = np.clip(p_arr, eps, 1.0 - eps)
    cand_brier = float(np.mean((p_arr - y_arr) ** 2))
    cand_log_loss = float(-np.mean(y_arr * np.log(p_safe) + (1 - y_arr) * np.log(1.0 - p_safe)))
    
    summary = {
        "candidate_rows": n_total,
        "n_non_empty_bins": int(len(df_non_empty)),
        "n_empty_bins": int(df_all["is_empty"].sum()),
        "overall_observed_win_rate": float(y_arr.mean()),
        "mean_predicted_probability": float(p_arr.mean()),
        "calibration_in_the_large": citl,
        "fixed_bin_ECE": ece,
        "fixed_bin_MCE": mce,
        "candidate_one_vs_rest_brier": cand_brier,
        "candidate_one_vs_rest_log_loss": cand_log_loss,
    }
    
    return df_all, df_non_empty, summary

df_all_bins, df_non_empty_bins, calibration_summary = compute_calibration_table(p, y)

print("=" * 75)
print("           EXP-0001-CAL-001 CALIBRATION SUMMARY METRICS")
print("=" * 75)
for k, v in calibration_summary.items():
    if isinstance(v, float):
        print(f"  {k:<32}: {v:.6f}")
    else:
        print(f"  {k:<32}: {v}")
print("=" * 75)

print("\n--- Non-Empty Reliability Bins (Contributing to ECE/MCE) ---")
print(df_non_empty_bins[["bin_label", "n_candidates", "weight", "mean_predicted_probability", "observed_win_frequency", "absolute_gap"]].to_string(index=False))

df_empty_bins = df_all_bins[df_all_bins["is_empty"]]
print(f"\n--- Diagnostic: Empty Bins ({len(df_empty_bins)}) Excluded from ECE/MCE ---")
print(df_empty_bins[["bin_label", "n_candidates", "is_empty"]].to_string(index=False))

           EXP-0001-CAL-001 CALIBRATION SUMMARY METRICS
  candidate_rows                  : 7285
  n_non_empty_bins                : 8
  n_empty_bins                    : 2
  overall_observed_win_rate       : 0.047495
  mean_predicted_probability      : 0.047495
  calibration_in_the_large        : 0.000000
  fixed_bin_ECE                   : 0.030667
  fixed_bin_MCE                   : 0.185189
  candidate_one_vs_rest_brier     : 0.040168
  candidate_one_vs_rest_log_loss  : 0.151746

--- Non-Empty Reliability Bins (Contributing to ECE/MCE) ---
   bin_label  n_candidates   weight  mean_predicted_probability  observed_win_frequency  absolute_gap
[0.00, 0.02)          1128 0.154839                    0.017009                0.000000      0.017009
[0.02, 0.04)          3336 0.457927                    0.028273                0.005396      0.022877
[0.04, 0.06)          1196 0.164173                    0.048663                0.035117      0.013546
[0.06, 0.08)           593 0.081400       

## 5. Exploratory Reliability Diagram & Sample Density Plot

We construct a 2-panel diagnostic figure:
- **Top Panel:** Observed win frequency vs. mean predicted probability across non-empty bins compared against the ideal $y = x$ diagonal line of perfect calibration. Empty bins are excluded to avoid showing artificial measurements.
- **Bottom Panel:** Log-scale candidate frequency histogram across probability bins, visualizing the empirical sample density.

In [5]:
# 5. Plot Candidate Reliability Diagram & Sample Density Histogram
fig, (ax1, ax2) = plt.subplots(
    nrows=2, ncols=1, figsize=(8.5, 7.5), sharex=True, gridspec_kw={"height_ratios": [3, 1.2]}
)

# Upper Panel: Reliability Curve (Non-empty bins only)
ax1.plot([0, 1], [0, 1], linestyle="--", color="gray", alpha=0.75, label="Perfect Calibration (y = x)")
ax1.plot(
    df_non_empty_bins["mean_predicted_probability"],
    df_non_empty_bins["observed_win_frequency"],
    marker="o",
    color="tab:red",
    linewidth=2,
    markersize=6,
    label=f"Driver Elo Baseline (ECE = {calibration_summary['fixed_bin_ECE']:.4f})",
)
ax1.set_title("EXP-0001 — Candidate-Level Reliability Diagram (2010–2026)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Observed Win Frequency", fontsize=10)
ax1.set_xlim(-0.02, 1.02)
ax1.set_ylim(-0.02, 1.02)
ax1.grid(True, alpha=0.25)
ax1.legend(loc="upper left", fontsize=9.5)

# Lower Panel: Sample Count Histogram (all bins)
bin_centers = [0.5 * (r["bin_lower"] + r["bin_upper"]) for _, r in df_all_bins.iterrows()]
bin_widths = [(r["bin_upper"] - r["bin_lower"]) * 0.88 for _, r in df_all_bins.iterrows()]
colors = ["tab:blue" if not r["is_empty"] else "lightgray" for _, r in df_all_bins.iterrows()]

ax2.bar(bin_centers, df_all_bins["n_candidates"], width=bin_widths, color=colors, edgecolor="black", alpha=0.7)
ax2.set_xlabel("Predicted Win Probability", fontsize=10)
ax2.set_ylabel("Candidates", fontsize=10)
ax2.set_yscale("log")
ax2.set_ylim(0.5, 10000)
ax2.grid(True, alpha=0.25, which="both")

plt.tight_layout()
plot_path = CAL_DIR / "candidate_reliability_diagram.png"
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Saved reliability diagram to: {plot_path.relative_to(PROJECT_ROOT)}")

Saved reliability diagram to: artifacts/reports/EXP-0001-CAL-001/candidate_reliability_diagram.png


## 6. Race-Clustered Bootstrap Uncertainty Intervals

### Statistical Methodology:
- Drivers in the same Grand Prix compete in a closed zero-sum contest: their probabilities sum to 1.0, and exactly one driver wins.
- A naive row-level bootstrap that resamples individual candidate rows violates intra-race constraints and produces spuriously narrow confidence intervals.
- **Race-Clustered Bootstrap Protocol:**
  1. Define the cluster unit as the complete Grand Prix event: `(season, round)` ($K = 346$ clusters).
  2. Resample $K = 346$ race clusters with replacement using a reproducible random seed (`seed = 42`).
  3. When a race cluster is sampled multiple times, all its entrant records are duplicated together as distinct independent realizations.
  4. Recompute ECE, MCE, candidate Brier score, candidate log loss, and CITL on each bootstrap replicate.
  5. Extract $95\%$ percentile confidence intervals ($[2.5\%, 97.5\%]$) across $R = 1,000$ iterations.

In [6]:
# 6. Execute Race-Clustered Bootstrap Uncertainty Estimation
def race_clustered_bootstrap(cal_df, n_replicates=1000, random_seed=42):
    rng = np.random.default_rng(random_seed)
    
    race_keys = cal_df[["season", "round"]].drop_duplicates().values
    n_races = len(race_keys)
    race_dict = {tuple(k): sub_df for k, sub_df in cal_df.groupby(["season", "round"])}
    
    boot_records = []
    
    for rep in range(n_replicates):
        sampled_indices = rng.choice(n_races, size=n_races, replace=True)
        # Treat repeated sampled races as separate clusters; keep candidate records together
        sampled_frames = [race_dict[tuple(race_keys[idx])] for idx in sampled_indices]
        boot_df = pd.concat(sampled_frames, ignore_index=True)
        
        b_p = boot_df["win_probability"].to_numpy(dtype=float)
        b_y = boot_df["y"].to_numpy(dtype=int)
        
        _, _, b_metrics = compute_calibration_table(b_p, b_y)
        boot_records.append(b_metrics)
        
    df_boot = pd.DataFrame(boot_records)
    
    summary_rows = []
    target_metrics = [
        "fixed_bin_ECE", "fixed_bin_MCE",
        "candidate_one_vs_rest_brier", "candidate_one_vs_rest_log_loss",
        "calibration_in_the_large"
    ]
    
    for m in target_metrics:
        vals = df_boot[m].to_numpy()
        summary_rows.append({
            "metric": m,
            "point_estimate": calibration_summary[m],
            "bootstrap_mean": float(np.mean(vals)),
            "bootstrap_std": float(np.std(vals)),
            "ci_95_lower": float(np.percentile(vals, 2.5)),
            "ci_95_upper": float(np.percentile(vals, 97.5)),
        })
        
    return pd.DataFrame(summary_rows), df_boot

print("Running race-clustered bootstrap (1,000 replicates, seed=42)... ~5-8 seconds.")
df_bootstrap_summary, df_bootstrap_replicates = race_clustered_bootstrap(cal, n_replicates=1000, random_seed=42)

print("\n" + "=" * 85)
print("           RACE-CLUSTERED BOOTSTRAP UNCERTAINTY INTERVALS (95% CI)")
print("=" * 85)
print(df_bootstrap_summary.to_string(index=False))
print("=" * 85)

Running race-clustered bootstrap (1,000 replicates, seed=42)... ~5-8 seconds.

           RACE-CLUSTERED BOOTSTRAP UNCERTAINTY INTERVALS (95% CI)
                        metric  point_estimate  bootstrap_mean  bootstrap_std  ci_95_lower  ci_95_upper
                 fixed_bin_ECE    3.066706e-02    3.061755e-02   1.802393e-03 2.679960e-02 3.415582e-02
                 fixed_bin_MCE    1.851888e-01    1.976780e-01   3.461403e-02 1.335617e-01 2.685974e-01
   candidate_one_vs_rest_brier    4.016764e-02    4.016066e-02   2.909167e-04 3.958892e-02 4.071968e-02
candidate_one_vs_rest_log_loss    1.517462e-01    1.517214e-01   1.426256e-03 1.490932e-01 1.545410e-01
      calibration_in_the_large    4.857226e-17    4.633793e-17   3.471639e-18 4.163336e-17 4.857226e-17


## 7. Descriptive Regulation Era & Season Calibration Stratification

We inspect whether calibration error drifts across Formula 1 technical eras:
- `2010_2013_v8_era` (77 races): V8 naturally aspirated engines, high competitive density.
- `2014_2021_hybrid_era` (160 races): Turbo-hybrid era, Mercedes dominance.
- `2022_2025_ground_effect_era` (92 races): Venturi tunnel ground-effect, Red Bull dominance.
- `2026_new_technical_era` (17 races YTD): Active season under new regulations.

> [!WARNING]
> **Small-Sample Notice for 2026:** The 2026 season comprises only 17 completed rounds to date ($N_{\text{races}} = 17$). Its calibration statistics carry high sampling variance and must be treated as preliminary and exploratory.

In [7]:
# 7. Stratified Calibration Analysis Across Regulation Eras & Seasons
era_records = []
for era, group in cal.groupby("regulation_era", sort=False):
    races_cnt = group[["season", "round"]].drop_duplicates().shape[0]
    cands_cnt = len(group)
    _, _, m = compute_calibration_table(group["win_probability"].to_numpy(), group["y"].to_numpy())
    era_records.append({
        "regulation_era": era,
        "races": races_cnt,
        "candidates": cands_cnt,
        "fixed_bin_ECE": m["fixed_bin_ECE"],
        "fixed_bin_MCE": m["fixed_bin_MCE"],
        "candidate_one_vs_rest_brier": m["candidate_one_vs_rest_brier"],
        "calibration_in_the_large": m["calibration_in_the_large"],
    })

df_era_calibration = pd.DataFrame(era_records)
print("=== Calibration Stratification Across Regulation Eras ===")
print(df_era_calibration.to_string(index=False))

# Season-by-season ECE evolution
season_records = []
for season, group in cal.groupby("season"):
    races_cnt = group[["season", "round"]].drop_duplicates().shape[0]
    _, _, m = compute_calibration_table(group["win_probability"].to_numpy(), group["y"].to_numpy())
    season_records.append({
        "season": season,
        "races": races_cnt,
        "candidates": len(group),
        "fixed_bin_ECE": m["fixed_bin_ECE"],
        "fixed_bin_MCE": m["fixed_bin_MCE"],
        "candidate_brier": m["candidate_one_vs_rest_brier"],
    })

df_season_calibration = pd.DataFrame(season_records)

# Plot season-by-season ECE drift
plt.figure(figsize=(9.5, 4.5))
plt.plot(df_season_calibration["season"], df_season_calibration["fixed_bin_ECE"], marker="o", color="tab:purple", linewidth=2, label="Season ECE")
plt.axvline(2013.5, color="gray", linestyle="--", alpha=0.6, label="Hybrid Shift (2014)")
plt.axvline(2021.5, color="gray", linestyle=":", alpha=0.6, label="Ground Effect (2022)")
plt.axvline(2025.5, color="gray", linestyle="-.", alpha=0.6, label="Technical Shift (2026)")
plt.title("Expected Calibration Error (ECE) Across Formula 1 Seasons (2010–2026)", fontsize=12, fontweight="bold")
plt.xlabel("Season", fontsize=10)
plt.ylabel("Fixed-Bin ECE (Lower is Better)", fontsize=10)
plt.grid(True, alpha=0.25)
plt.legend(loc="upper right", fontsize=9)
plt.tight_layout()
season_plot_path = CAL_DIR / "season_ece_drift.png"
plt.savefig(season_plot_path, dpi=150)
plt.show()
print(f"Saved season ECE drift plot to: {season_plot_path.relative_to(PROJECT_ROOT)}")

=== Calibration Stratification Across Regulation Eras ===
             regulation_era  races  candidates  fixed_bin_ECE  fixed_bin_MCE  candidate_one_vs_rest_brier  calibration_in_the_large
           2010_2013_v8_era     77        1806       0.034530       0.255801                     0.038294              4.857226e-17
       2014_2021_hybrid_era    160        3267       0.037442       0.277595                     0.039826              4.857226e-17
2022_2025_ground_effect_era     92        1838       0.026250       0.233456                     0.042139              4.857226e-17
     2026_new_technical_era     17         374       0.019576       0.079265                     0.042517              4.857226e-17
Saved season ECE drift plot to: artifacts/reports/EXP-0001-CAL-001/season_ece_drift.png


## 8. Serialization & Experiment Manifest Protocol

We serialize the complete calibration diagnostic suite and experiment manifest to `artifacts/reports/EXP-0001-CAL-001/`.

In [8]:
# 8. Save Artifacts & Generate Experiment Manifest
summary_path = CAL_DIR / "candidate_calibration_summary.csv"
bins_path = CAL_DIR / "candidate_reliability_bins.csv"
empty_bins_path = CAL_DIR / "empty_bins_diagnostic.csv"
bootstrap_path = CAL_DIR / "bootstrap_uncertainty_summary.csv"
era_path = CAL_DIR / "regulation_era_calibration.csv"
season_path = CAL_DIR / "season_calibration.csv"
manifest_path = CAL_DIR / "experiment_manifest.json"

# Serialize CSV tables
pd.DataFrame([calibration_summary]).to_csv(summary_path, index=False)
df_non_empty_bins.to_csv(bins_path, index=False)
df_empty_bins.to_csv(empty_bins_path, index=False)
df_bootstrap_summary.to_csv(bootstrap_path, index=False)
df_era_calibration.to_csv(era_path, index=False)
df_season_calibration.to_csv(season_path, index=False)

# Prepare structured experiment manifest
manifest = {
    "experiment_id": "EXP-0001-CAL-001",
    "status": "COMPLETED_EXPLORATORY",
    "source_candidate_predictions": str(SOURCE_PATH.relative_to(PROJECT_ROOT)),
    "source_version": SOURCE_VERSION,
    "candidate_rows": int(len(cal)),
    "race_count": int(len(per_race)),
    "max_race_probability_sum_error": max_sum_error,
    "calibration_method": "Fixed-bin candidate-level one-vs-rest (empty bins excluded from ECE/MCE)",
    "bin_edges": BIN_EDGES.tolist(),
    "boundary_convention": "half-open [bin_lower, bin_upper) with rightmost closed at 1.0",
    "metrics": calibration_summary,
    "bootstrap_settings": {
        "cluster_unit": "(season, round) Grand Prix event",
        "replicates": 1000,
        "random_seed": 42,
        "confidence_interval_level": 0.95,
    },
    "bootstrap_summary": df_bootstrap_summary.set_index("metric").to_dict(orient="index"),
    "artifacts": [
        str(summary_path.relative_to(PROJECT_ROOT)),
        str(bins_path.relative_to(PROJECT_ROOT)),
        str(empty_bins_path.relative_to(PROJECT_ROOT)),
        str(bootstrap_path.relative_to(PROJECT_ROOT)),
        str(era_path.relative_to(PROJECT_ROOT)),
        str(season_path.relative_to(PROJECT_ROOT)),
        str((CAL_DIR / "candidate_reliability_diagram.png").relative_to(PROJECT_ROOT)),
        str((CAL_DIR / "season_ece_drift.png").relative_to(PROJECT_ROOT)),
    ],
    "limitations": [
        "Candidate set is reconstructed retrospectively from race result records rather than verified pre-race entry lists.",
        "Drivers within a race are zero-sum competitors; row-level independent uncertainty is statistically invalid.",
        "Fixed-bin ECE is sensitive to pre-specified bin boundary choices.",
        "2026 season comprises only 17 rounds and carries high sampling variance.",
        "This audit evaluates raw uncalibrated baseline outputs prior to fitting post-hoc recalibration models.",
    ],
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Serialized all calibration artifacts to EXP-0001-CAL-001:")
for p in [
    summary_path, bins_path, empty_bins_path, bootstrap_path,
    era_path, season_path, CAL_DIR / "candidate_reliability_diagram.png",
    CAL_DIR / "season_ece_drift.png", manifest_path
]:
    print(f"  - {p.name} ({p.stat().st_size:,} bytes)")

Serialized all calibration artifacts to EXP-0001-CAL-001:
  - candidate_calibration_summary.csv (362 bytes)
  - candidate_reliability_bins.csv (1,227 bytes)
  - empty_bins_diagnostic.csv (228 bytes)
  - bootstrap_uncertainty_summary.csv (702 bytes)
  - regulation_era_calibration.csv (562 bytes)
  - season_calibration.csv (1,298 bytes)
  - candidate_reliability_diagram.png (91,210 bytes)
  - season_ece_drift.png (101,586 bytes)
  - experiment_manifest.json (3,704 bytes)


## 9. Scientific Discussion & Handoff to Recalibration

### Summary of Findings (EXP-0001-CAL-001):
1. **Calibration-in-the-Large Balance:** Overall observed win frequency ($0.047495$) matches overall mean predicted win probability ($0.047495$) across the 7,285 entrant records, confirming global calibration balance ($\text{CITL} = 0.000000$).
2. **Expected Calibration Error:** The raw driver-only Elo baseline achieves a fixed-bin $\text{ECE} = 0.030667$ ($95\%$ race-clustered CI: $[0.026800, 0.034156]$) across the 8 non-empty bins.
3. **Empty Bins Diagnostics:** No candidates in the baseline dataset received win probabilities $\ge 0.30$. Bins $[0.30, 0.50)$ and $[0.50, 1.00]$ are properly excluded from ECE/MCE calculations.
4. **Maximum Calibration Gap:** $\text{MCE} = 0.185189$ ($95\%$ CI: $[0.133562, 0.268597]$) occurs in the $[0.15, 0.20)$ bin, where top contenders often win more frequently than predicted by a driver-alone model without constructor advantage.

### Handoff to Post-Hoc Recalibration (Milestone 4):
The observed miscalibration in higher probability bins motivates post-hoc calibration techniques (Isotonic Regression, Platt Scaling, Temperature Scaling) to be evaluated on rolling validation folds in `TSK-MOD-005`.